In [1]:
import pandas as pd
import os

data_path = "../data/raw/"
files = [
    "olist_customers_dataset.csv", "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv", "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv", "olist_orders_dataset.csv",
    "olist_products_dataset.csv", "olist_sellers_dataset.csv",
    "product_category_name_translation.csv",
]
dfs = {}
for f in files:
    name = f.replace("olist_", "").replace("_dataset.csv", "").replace(".csv", "")
    dfs[name] = pd.read_csv(os.path.join(data_path, f))

Fix data types — convert all date columns to datetime

In [3]:
date_cols = {
    "orders": ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
               "order_delivered_customer_date", "order_estimated_delivery_date"],
    "order_items": ["shipping_limit_date"],
    "order_reviews": ["review_creation_date", "review_answer_timestamp"],
}
for table, cols in date_cols.items():
    for col in cols:
        dfs[table][col] = pd.to_datetime(dfs[table][col])

Clean geolocation duplicates

In [4]:
dfs["geolocation"] = dfs["geolocation"].drop_duplicates()
geo_clean = dfs["geolocation"].groupby("geolocation_zip_code_prefix").agg({
    "geolocation_lat": "mean",
    "geolocation_lng": "mean",
    "geolocation_city": "first",
    "geolocation_state": "first",
}).reset_index()
dfs["geolocation"] = geo_clean

Fix invalid payment_type

In [5]:
dfs["order_payments"] = dfs["order_payments"][dfs["order_payments"]["payment_type"] != "not_defined"]

Handle products with missing metadata

In [10]:
dfs["products"]["product_category_name"] = dfs["products"]["product_category_name"].fillna("unknown")

Fix invalid product_weight_g (0g)

In [6]:
median_weight = dfs["products"].loc[dfs["products"]["product_weight_g"] > 0, "product_weight_g"].median()
dfs["products"]["product_weight_g"] = dfs["products"]["product_weight_g"].replace(0, median_weight)

Rename columns for clarity

In [7]:
dfs["products"].rename(columns={"product_name_lenght": "product_name_length",
                                  "product_description_lenght": "product_description_length"}, inplace=True)

Save cleaned data

In [11]:
os.makedirs("../data/cleaned", exist_ok=True)
for name, df in dfs.items():
    df.to_csv(f"../data/cleaned/{name}_cleaned.csv", index=False)
print("All cleaned files saved.")

All cleaned files saved.


In [12]:
   dfs["geolocation"].shape

(19015, 5)

In [13]:
   dfs["order_payments"]["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
Name: count, dtype: int64

In [14]:
   dfs["products"]["product_category_name"].isnull().sum()

np.int64(0)

In [15]:
   (dfs["products"]["product_weight_g"] == 0).sum()

np.int64(0)